In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import requests
import joblib

import numpy, pandas
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [3]:
train = pandas.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pandas.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

In [4]:
def tokenize_smiles(smiles):
    pattern = r"(\[[^\[\]]{1,6}\])"
    tokens = re.split(pattern, smiles)
    result = []
    for token in tokens:
        if token.startswith('['):
            result.append(token)
        else:
            result.extend(list(token))
    return result


#print(max([len(i) for i in train['SMILES']]))
train_token = [tokenize_smiles(i) for i in train['SMILES']]
train_token = numpy.array([i+([0]*(306-len(i))) for i in train_token])

train_token_df = pandas.DataFrame({f'c_{j}' : numpy.array([train_token[i][j] for i in range(len(train_token))]) for j in range(306)})

test_token = [tokenize_smiles(i) for i in test['SMILES']]
test_token = numpy.array([i+([0]*(306-len(i))) for i in test_token])

test_token_df = pandas.DataFrame({f'c_{j}' : numpy.array([test_token[i][j] for i in range(len(test_token))]) for j in range(306)})

In [5]:
train = pandas.concat([train, train_token_df], axis=1).drop(columns=['SMILES'])
test = pandas.concat([test, test_token_df], axis=1).drop(columns=['SMILES'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(train.columns, train.dtypes):
    if i[0] != 'id':
        if i[1]=='O':
            encoder.fit(train[i[0]])
            
            train[i[0]] = train[i[0]].fillna('Unknown')
            train[i[0]] = encoder.transform(train[i[0]].to_numpy().reshape(-1,1))
    
            test[i[0]] = test[i[0]].fillna('Unknown')
            test[i[0]] = numpy.array(encoder.transform(test[i[0]].to_numpy().reshape(-1,1)))
        else:
            normalize.fit(numpy.array(train[i[0]]).reshape(-1,1))
            
            train[i[0]].fillna(0, inplace=True)
            train[i[0]] = normalize.transform(numpy.array(train[i[0]]).reshape(-1,1))
            
            if i[0] in test.columns:
                test[i[0]].fillna(0, inplace=True)
                test[i[0]] = normalize.transform(numpy.array(test[i[0]]).reshape(-1,1))

In [7]:
train

,id,Tg,FFV,Tc,Density,Rg,c_0,c_1,c_2,c_3,...,c_296,c_297,c_298,c_299,c_300,c_301,c_302,c_303,c_304,c_305
0,87817,0.000000,0.374645,0.205667,0.0,0.0,0,2,6,1,...,1,0,0,1,0,0,0,1,0,0
1,106919,0.000000,0.370410,0.000000,0.0,0.0,0,3,18,5,...,1,0,0,1,0,0,0,1,0,0
2,388772,0.000000,0.378860,0.000000,0.0,0.0,0,4,18,5,...,1,0,0,1,0,0,0,1,0,0
3,519416,0.000000,0.387324,0.000000,0.0,0.0,0,3,18,5,...,1,0,0,1,0,0,0,1,0,0
4,539187,0.000000,0.355470,0.000000,0.0,0.0,0,4,18,5,...,1,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7968,2146592435,0.000000,0.367498,0.000000,0.0,0.0,0,4,18,5,...,1,0,0,1,0,0,0,1,0,0
7969,2146810552,0.000000,0.353280,0.000000,0.0,0.0,0,2,1,6,...,1,0,0,1,0,0,0,1,0,0
7970,2147191531,0.000000,0.369411,0.000000,0.0,0.0,0,8,4,19,...,1,0,0,1,0,0,0,1,0,0
7971,2147435020,261.662355,0.000000,0.000000,0.0,0.0,0,2,5,7,...,1,0,0,1,0,0,0,1,0,0


In [8]:
test

,id,c_0,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,...,c_296,c_297,c_298,c_299,c_300,c_301,c_302,c_303,c_304,c_305
0,1109053969,0,4,18,5,20,20,25,1,12,...,1,0,0,1,0,0,0,1,0,0
1,1422188626,0,4,18,5,20,20,25,1,12,...,1,0,0,1,0,0,0,1,0,0
2,2032016830,0,8,4,19,20,20,25,1,15,...,1,0,0,1,0,0,0,1,0,0


In [9]:
train_c_x = train[[f'c_{i}' for i in range(306)]]
train_c_y = train[['Tg', 'FFV', 'Tc', 'Density', 'Rg']]
X_train, X_test, y_train, y_test = train_test_split(train_c_x, train_c_y, 
                                                    test_size=0.07, random_state=100)

In [10]:
catboost_params = {'iterations' : 3000,
                   'learning_rate': 0.009, 
                   'depth': 5, 
                   'l2_leaf_reg': 5.5,
                   'min_child_samples' : 102,
                   'od_wait' : 50,
                   'random_state' : 42,
                   'od_type' : 'Iter',
                   'bootstrap_type': 'Bayesian', 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent',
                   'loss_function': 'MultiRMSE'}

catboost = CatBoostRegressor(**catboost_params)

cat_features = list(X_train.select_dtypes(include=['object', 'category']).columns)
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)

catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)

In [11]:
test_pre = catboost.predict(test[[f'c_{i}' for i in range(306)]])
submission = {'id' : [], 'Tg' : [], 'FFV' : [], 'Tc' : [], 'Density' : [], 'Rg' : []}
for i, p in zip(test['id'], test_pre):
    submission['id'].append(i)
    submission['Tg'].append(p[0])
    submission['FFV'].append(p[1])
    submission['Tc'].append(p[2])
    submission['Density'].append(p[3])
    submission['Rg'].append(p[4])

pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)